## LangGraph vacancy classifier

A compact demonstration of shared state, tool-backed nodes, conditional edges, retries, async model calls, and provider-independent orchestration. The graph is deterministic: LangGraph selects the tools, the LLM only performs classification.

In [ ]:
import json
import os
import time
from enum import Enum
from pathlib import Path
from typing import Literal, TypedDict

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.tools import tool
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_groq import ChatGroq
from langchain_ollama import ChatOllama
from langgraph.graph import END, START, StateGraph
from pydantic import BaseModel, Field

from langchain_mcp_adapters.client import MultiServerMCPClient

### Models and schema

In [ ]:
GROQ_API_KEY = ""
GEMINI_API_KEY = ""


CATEGORIES = [
    "Data Scientist",
    "Data Analyst",
    "Data Engineer",
    "MLOps Engineer",
    "Business Analyst",
    "BI Developer",
]

Category = Literal[
    "Data Scientist", "Data Analyst", "Data Engineer",
    "MLOps Engineer", "Business Analyst", "BI Developer",
]

class JobType(str, Enum):
    PROJECT = "project-based"
    PERMANENT = "permanent"

class SearchType(str, Enum):
    LOOKING_FOR_WORK = "looking for work"
    LOOKING_FOR_EMPLOYEE = "looking for employee"

class ClassificationResult(BaseModel):
    job_type: JobType = Field(description="Whether the role is permanent or project-based")
    category: Category = Field(description="The closest profession category")
    search_type: SearchType = Field(description="Whether the author seeks work or an employee")

class WorkflowState(TypedDict):
    input_path: str
    output_path: str
    model_name: str
    jobs: list[dict]
    current_index: int
    current_job: dict
    description: str
    job_type: str
    category: str
    search_type: str
    results: list[dict]
    retry_count: int
    max_retries: int
    validation_error: str
    started_at: float

In [ ]:
def make_llm(provider: str = "ollama"):
    """change the provider without changing graph logic"""
    if provider == "ollama":
        return ChatOllama(model=os.getenv("OLLAMA_MODEL", "gemma4:12b-mlx"), temperature=0)
    if provider == "groq":
        return ChatGroq(
            model=os.getenv("GROQ_MODEL", "llama-3.3-70b-versatile"),
            api_key=GROQ_API_KEY,
            temperature=0,
        )
    if provider == "gemini":
        return ChatGoogleGenerativeAI(
            model=os.getenv("GEMINI_MODEL", "gemini-flash-lite-latest"),
            api_key=GEMINI_API_KEY,
            temperature=0,
        )
    raise ValueError(f"Unknown provider: {provider}")

### Tools

Add the local tools are and the optional MCP as fallback.

In [ ]:
@tool
def load_jobs_data(path: str) -> list[dict]:
    """load job descriptions from a JSON file"""
    with Path(path).open("r", encoding="utf-8") as file:
        data = json.load(file)
    if not isinstance(data, list) or not data:
        raise ValueError("Input JSON must contain a non-empty list of jobs")
    if any(not isinstance(job, dict) or not job.get("description") for job in data):
        raise ValueError("Every job must be an object with a non-empty description")
    return data

@tool
def save_classification_results(path: str, results: list[dict]) -> dict:
    """overwrite a JSONL file with classification results"""
    output = Path(path)
    output.parent.mkdir(parents=True, exist_ok=True)
    with output.open("w", encoding="utf-8") as file:
        for result in results:
            file.write(json.dumps(result, ensure_ascii=False) + "\n")
    return {"path": str(output), "saved": len(results)}

class StorageBackend:
    def __init__(self, use_mcp: bool = False):
        self.use_mcp = use_mcp
        self.mcp_tools = {}

    async def connect(self):
        if not self.use_mcp:
            return self
        client = MultiServerMCPClient({
            "filesystem": {
                "command": "npx",
                "args": ["-y", "@modelcontextprotocol/server-filesystem", str(Path.cwd())],
                "transport": "stdio",
            }
        })
        self.mcp_tools = {item.name: item for item in await client.get_tools()}
        return self

    async def load(self, path: str) -> list[dict]:
        if not self.use_mcp:
            return await load_jobs_data.ainvoke({"path": path})
        raw = await self.mcp_tools["read_text_file"].ainvoke({"path": str(Path(path).resolve())})
        text = raw if isinstance(raw, str) else raw.get("content", raw)
        data = json.loads(text)
        if not isinstance(data, list) or not data:
            raise ValueError("Input JSON must contain a non-empty list of jobs")
        return data

    async def save(self, path: str, results: list[dict]) -> dict:
        if not self.use_mcp:
            return await save_classification_results.ainvoke({"path": path, "results": results})
        content = "".join(json.dumps(row, ensure_ascii=False) + "\n" for row in results)
        await self.mcp_tools["write_file"].ainvoke({
            "path": str(Path(path).resolve()), "content": content
        })
        return {"path": path, "saved": len(results)}

### Graph

We will find the file with data and load it into memory. Each job description is classified with one structured LLM call. Conditional edges implement both bounded retries and iteration over the batch.

In [ ]:
class VacancyClassificationWorkflow:
    def __init__(self, llm, storage: StorageBackend):
        self.llm = llm
        self.storage = storage
        self.model_name = (
            getattr(llm, "model", None)
            or getattr(llm, "model_name", None)
            or getattr(llm, "model_id", None)
            or "unknown_model"
        )
        self.graph = self._build_graph()

    def _build_graph(self):
        graph = StateGraph(WorkflowState)
        graph.add_node("load_jobs", self._load_jobs)
        graph.add_node("select_job", self._select_job)
        graph.add_node("classify", self._classify)
        graph.add_node("collect_result", self._collect_result)
        graph.add_node("save_results", self._save_results)

        graph.add_edge(START, "load_jobs")
        graph.add_edge("load_jobs", "select_job")
        graph.add_edge("select_job", "classify")
        graph.add_conditional_edges(
            "classify", self._classification_route,
            {"retry": "classify", "collect": "collect_result"},
        )
        graph.add_conditional_edges(
            "collect_result", self._batch_route,
            {"next": "select_job", "save": "save_results"},
        )
        graph.add_edge("save_results", END)
        return graph.compile()

    async def _load_jobs(self, state: WorkflowState):
        jobs = await self.storage.load(state["input_path"])
        return {"jobs": jobs, "current_index": 0, "results": []}

    async def _select_job(self, state: WorkflowState):
        job = state["jobs"][state["current_index"]]
        return {
            "current_job": job, "description": job["description"],
            "job_type": "", "category": "", "search_type": "",
            "retry_count": 0, "validation_error": "",
        }

    async def _classify(self, state: WorkflowState):
        prompt = ChatPromptTemplate.from_messages([
            ("system", """Classify the job description by three dimensions.
            Job type: project-based for temporary, freelance, part-time or one-time work;
            otherwise permanent.
            Category: select the closest option from {categories}.
            Search type: looking for work when the author seeks a job or project;
            looking for employee when the author is hiring."""),
            ("human", "{description}"),
        ])
        try:
            messages = prompt.invoke({
                "description": state["description"],
                "categories": ", ".join(CATEGORIES),
            })
            response = await self.llm.with_structured_output(
                ClassificationResult, include_raw=True
            ).ainvoke(messages)
            result = response["parsed"]
            if result is None:
                result = self._parse_text_fallback(self._get_text(response["raw"]))
            if result is None:
                raise response["parsing_error"] or ValueError("Could not parse model output")
            if not isinstance(result, ClassificationResult):
                result = ClassificationResult.model_validate(result)
            return {
                "job_type": result.job_type.value,
                "category": result.category,
                "search_type": result.search_type.value,
                "validation_error": "",
            }
        except Exception as exc:
            return {
                "job_type": "", "category": "", "search_type": "",
                "retry_count": state["retry_count"] + 1,
                "validation_error": str(exc),
            }

    @staticmethod
    def _get_text(response) -> str:
        if isinstance(response.content, str):
            return response.content
        return "".join(
            block.get("text", "")
            for block in response.content
            if isinstance(block, dict) and block.get("type") == "text"
        )

    @staticmethod
    def _parse_text_fallback(text: str):
        """recover known labels when a local model ignores the JSON schema"""
        normalized = " ".join(text.replace("**", "").lower().split())

        if any(label in normalized for label in ("project-based", "project job", "freelance", "part-time")):
            job_type = JobType.PROJECT
        elif "permanent" in normalized:
            job_type = JobType.PERMANENT
        else:
            return None

        category = next(
            (item for item in CATEGORIES if item.lower() in normalized), None
        )
        if category is None:
            return None

        if "looking for employee" in normalized:
            search_type = SearchType.LOOKING_FOR_EMPLOYEE
        elif "looking for work" in normalized:
            search_type = SearchType.LOOKING_FOR_WORK
        else:
            return None

        return ClassificationResult(
            job_type=job_type, category=category, search_type=search_type
        )

    def _classification_route(self, state: WorkflowState):
        if state["validation_error"] and state["retry_count"] <= state["max_retries"]:
            return "retry"
        return "collect"

    async def _collect_result(self, state: WorkflowState):
        row = {
            "index": state["current_index"],
            "description": state["description"],
            "source_role": state["current_job"].get("source_role"),
            "expected": state["current_job"]["expected"],
            "prediction": {
                "job_type": state["job_type"] or "UNKNOWN",
                "category": state["category"] or "UNKNOWN",
                "search_type": state["search_type"] or "UNKNOWN",
            },
            "model": state["model_name"],
            "retries": min(state["retry_count"], state["max_retries"]),
            "error": state["validation_error"] or None,
        }
        return {
            "results": [*state["results"], row],
            "current_index": state["current_index"] + 1,
        }

    def _batch_route(self, state: WorkflowState):
        return "next" if state["current_index"] < len(state["jobs"]) else "save"

    async def _save_results(self, state: WorkflowState):
        await self.storage.save(state["output_path"], state["results"])
        return {}

    async def run(self, input_path="test.json", output_path="classification_results.jsonl", max_retries=1):
        if max_retries < 0:
            raise ValueError("max_retries must be non-negative")
        initial_state = {
            "input_path": input_path, "output_path": output_path,
            "model_name": str(self.model_name), "jobs": [],
            "current_index": 0, "current_job": {}, "description": "",
            "job_type": "", "category": "", "search_type": "",
            "results": [], "retry_count": 0, "max_retries": max_retries,
            "validation_error": "", "started_at": time.perf_counter(),
        }
        
        return await self.graph.ainvoke(
            initial_state, config={"recursion_limit": 10_000}
        )

In [ ]:
# here choose LLM: ollama, groq or gemini
PROVIDER = "ollama"
USE_MCP_FILESYSTEM = False

storage = await StorageBackend(use_mcp=USE_MCP_FILESYSTEM).connect()
workflow = VacancyClassificationWorkflow(make_llm(PROVIDER), storage)
result = await workflow.run(
    input_path="test.json",
    output_path=f"classification_results_{PROVIDER}.jsonl",
    max_retries=1,
)

rows = result["results"]
fields = ("job_type", "category", "search_type")

# lets check accuracy of forecasts
field_accuracy = {
    field: sum(row["prediction"][field] == row["expected"][field] for row in rows) / len(rows)
    for field in fields
}
joint_accuracy = sum(
    all(row["prediction"][field] == row["expected"][field] for field in fields)
    for row in rows
) / len(rows)

print(f"Processed: {len(rows)}")
print(f"Errors: {sum(row['error'] is not None for row in rows)}")
for field, score in field_accuracy.items():
    print(f"{field} accuracy: {score:.1%}")
print(f"Joint accuracy: {joint_accuracy:.1%}")
print(f"Elapsed: {time.perf_counter() - result['started_at']:.2f}s")
result["results"][:2]

Processed: 5
Errors: 0
job_type accuracy: 100.0%
category accuracy: 80.0%
search_type accuracy: 100.0%
Joint accuracy: 80.0%
Elapsed: 676.52s


[{'index': 0,
  'description': 'The Data Science Team within Customer Success is highly engaged with our clients making use of their critical thinking skills with a business-focused mentality and customer-facing attitude. They activate, maintain, and support clients, develop models and rules, and train & enable them. In addition, they work cross-functionally with other departments (e.g., Research, Product, Marketing) in a collaborative team spirit spanning the globe to ensure we deliver best in class risk prevention solutions. Being on the frontline of fighting crime and protecting people from financial harm is incredibly inspiring to each of us. Join Us!\nYour Day To Day\nUnderstanding the data which our clients provide to us;\nCleaning that data and validating that it is correct;\nPreprocessing the data, usually by using a mixture of shell scripts and a programming language such as Python, Java, Scala, etc\nIteratively computing features and tuning parameters to improve the quality o